# MAPPO

In [8]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta
from torch.optim import Adam
from collections import defaultdict, deque
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ZINB_H1_PARAMS = [
    {"phi": 0.25, "p": 0.48, "n": 15},
]

ZINB_H2_PARAMS = [
    {"phi": 0.25, "p": 0.48, "n": 15},
]

def _zinb_mean(params: dict) -> float:
    phi, p, n = params["phi"], params["p"], params["n"]
    return (1.0 - phi) * n * (1.0 - p) / p

def sample_zinb(phi: float, p: float, n: int) -> int:
    if np.random.random() < phi: return 0
    return int(np.random.negative_binomial(n, p))

def sample_hospital_demand(zinb_params: list, mult: float = 1.0) -> int:
    total = sum(sample_zinb(p["phi"], p["p"], p["n"]) for p in zinb_params)
    return max(0, int(total * mult))

# Base config - will be modified by sensitivity analysis
BASE_ENV_CONFIG = {
    "max_capacity": 100,
    "shelf_life": 5,
    "order_cost": 1.0,
    "transshipment_cost": 1.0,
    "transshipment_fixed_cost": 5.0,
    "shortage_cost": 15.0,
    "wastage_cost": 15.0,
    "inventory_cost": 0.5,
    "episode_length": 60,
    "order_cap": 70,
    "trans_cap": 50,
    "transshipment_deadband": 0.1,
    "disruption_threshold": 1.2,
    "disruption_prob": 0.02,
    "disruption_min_len": 3,
    "disruption_max_len": 8,
    "disruption_mult": 1.4,
}

ORDER_SCHEDULE_H1 = {1, 4, 6}
ORDER_SCHEDULE_H2 = {0, 3, 5}

STATE_DIM = 26
PROCUREMENT_ACTION_DIM = 2
LOGISTICS_ACTION_DIM = 1
STATE_SCALE = 70.0
REWARD_SCALE = 100.0

# MAPPO Hyperparameters
LR = 3e-4
GAMMA = 0.99
EPSILON = 0.2
EPOCHS = 4
GAE_LAMBDA = 0.95
UPDATE_EPISODES = 4
ENTROPY_COEF = 0.05
ENTROPY_DECAY = 0.999
MIN_ENTROPY = 0.005
TOTAL_EPISODES = 2000

MODEL_PATH = "mappo.pt"
OUTPUT_PREFIX = "mappo"

# =============================================================================
# ENVIRONMENT
# =============================================================================
class PlateletSupplyChainEnv:
    def __init__(self, env_config=None):
        if env_config is None:
            env_config = BASE_ENV_CONFIG
        for k, v in env_config.items():
            setattr(self, k, v)
        self.num_hospitals = 2
        self.demand_mean_h1 = sum(_zinb_mean(p) for p in ZINB_H1_PARAMS)
        self.demand_mean_h2 = sum(_zinb_mean(p) for p in ZINB_H2_PARAMS)
        self.reset()

    def _is_order_day_h1(self): return (self.day % 7) in ORDER_SCHEDULE_H1
    def _is_order_day_h2(self): return (self.day % 7) in ORDER_SCHEDULE_H2

    def _compute_disruption_flag(self, history):
        if len(history) < 5: return 0.0
        ma2 = np.mean(list(history)[-2:])
        ma30 = np.mean(list(history)[-30:])
        return 1.0 if (ma30 > 0 and ma2 > self.disruption_threshold * ma30) else 0.0

    def _get_observation(self):
        obs = np.zeros((STATE_DIM,), dtype=np.float32)
        idx = 0
        for h in range(self.num_hospitals):
            start = h * self.shelf_life
            for age in range(self.shelf_life):
                obs[idx] = self.state[start + age]
                idx += 1

        inv_h1 = np.sum(self.state[0:self.shelf_life])
        inv_h2 = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        obs[10] = self.day / self.episode_length
        obs[11] = self._compute_disruption_flag(self.demand_history_h1)
        obs[12] = self._compute_disruption_flag(self.demand_history_h2)
        obs[13] = inv_h1 / self.max_capacity
        obs[14] = inv_h2 / self.max_capacity
        obs[15] = 1.0 if self._is_order_day_h1() else 0.0
        obs[16] = 1.0 if self._is_order_day_h2() else 0.0
        obs[17] = (inv_h1 - inv_h2) / self.max_capacity
        obs[18] = self.crisis_counter_h1 / self.disruption_max_len
        obs[19] = self.crisis_counter_h2 / self.disruption_max_len

        d1 = np.mean(self.demand_history_h1[-7:]) if self.demand_history_h1 else self.demand_mean_h1
        d2 = np.mean(self.demand_history_h2[-7:]) if self.demand_history_h2 else self.demand_mean_h2
        obs[20] = min(inv_h1 / (d1 * 7 + 1e-8), 3.0)
        obs[21] = min(inv_h2 / (d2 * 7 + 1e-8), 3.0)

        if self.demand_history_h1:
            td1 = sum(self.demand_history_h1[-30:])
            obs[22] = sum(self.shortage_history_h1) / (td1 + 1e-8)
            obs[24] = sum(self.wastage_history_h1) / (td1 + 1e-8)
        if self.demand_history_h2:
            td2 = sum(self.demand_history_h2[-30:])
            obs[23] = sum(self.shortage_history_h2) / (td2 + 1e-8)
            obs[25] = sum(self.wastage_history_h2) / (td2 + 1e-8)
        return obs.copy()

    def reset(self):
        self.state = np.zeros((self.num_hospitals * self.shelf_life,), dtype=np.float32)
        self.day = 0
        self.demand_history_h1, self.demand_history_h2 = [], []
        self.shortage_history_h1, self.shortage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.wastage_history_h1, self.wastage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.disruption_counter_h1 = self.disruption_counter_h2 = 0
        self.crisis_counter_h1 = self.crisis_counter_h2 = 0
        return self._get_observation()

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        is_ord_h1 = self._is_order_day_h1()
        is_ord_h2 = self._is_order_day_h2()

        order_h1 = action[0] * self.order_cap if is_ord_h1 else 0.0
        order_h2 = action[1] * self.order_cap if is_ord_h2 else 0.0

        a_t = action[2] * 2.0 - 1.0
        tau = self.transshipment_deadband

        trans_h1_to_h2 = trans_h2_to_h1 = 0.0
        if a_t > tau:
            trans_h1_to_h2 = ((a_t - tau) / (1.0 - tau)) * self.trans_cap
        elif a_t < -tau:
            trans_h2_to_h1 = ((abs(a_t) - tau) / (1.0 - tau)) * self.trans_cap

        inv_h1_before = np.sum(self.state[0:self.shelf_life])
        inv_h2_before = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        actual_order_h1 = self._process_order(0, min(order_h1, max(0.0, self.max_capacity - inv_h1_before)))
        actual_order_h2 = self._process_order(1, min(order_h2, max(0.0, self.max_capacity - inv_h2_before)))

        actual_trans_h1_to_h2 = self._transfer_blood(0, 1, trans_h1_to_h2)
        actual_trans_h2_to_h1 = self._transfer_blood(1, 0, trans_h2_to_h1)

        if self.day >= 5:
            if self.disruption_counter_h1 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h1 = random.randint(self.disruption_min_len, self.disruption_max_len)
            if self.disruption_counter_h2 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h2 = random.randint(self.disruption_min_len, self.disruption_max_len)

        mult_h1 = self.disruption_mult if self.disruption_counter_h1 > 0 else 1.0
        mult_h2 = self.disruption_mult if self.disruption_counter_h2 > 0 else 1.0

        if self.disruption_counter_h1 > 0: self.disruption_counter_h1 -= 1
        if self.disruption_counter_h2 > 0: self.disruption_counter_h2 -= 1

        demand_h1 = sample_hospital_demand(ZINB_H1_PARAMS, mult=mult_h1)
        demand_h2 = sample_hospital_demand(ZINB_H2_PARAMS, mult=mult_h2)

        self.demand_history_h1.append(demand_h1)
        self.demand_history_h2.append(demand_h2)

        self.crisis_counter_h1 = min(self.crisis_counter_h1 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h1) > 0.5 else 0
        self.crisis_counter_h2 = min(self.crisis_counter_h2 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h2) > 0.5 else 0

        wastage_h1, shortage_h1, _ = self._fulfill_demand(0, demand_h1)
        wastage_h2, shortage_h2, _ = self._fulfill_demand(1, demand_h2)

        self.shortage_history_h1.append(shortage_h1)
        self.shortage_history_h2.append(shortage_h2)
        self.wastage_history_h1.append(wastage_h1)
        self.wastage_history_h2.append(wastage_h2)

        self._age_blood()

        inv_h1_after = np.sum(self.state[0:self.shelf_life])
        inv_h2_after = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        inventory_holding_cost = (inv_h1_before + inv_h2_before) * self.inventory_cost

        self.day += 1

        total_transferred = actual_trans_h1_to_h2 + actual_trans_h2_to_h1
        transshipment_cost = total_transferred * self.transshipment_cost + (self.transshipment_fixed_cost if total_transferred >= 1.0 else 0.0)
        wastage_cost = (wastage_h1 + wastage_h2) * self.wastage_cost
        shortage_cost = (shortage_h1 + shortage_h2) * self.shortage_cost
        order_cost = (actual_order_h1 + actual_order_h2) * self.order_cost

        total_cost = wastage_cost + shortage_cost + transshipment_cost + order_cost + inventory_holding_cost
        shared_reward = -total_cost / REWARD_SCALE

        info = {
            'order_mask': [1.0 if is_ord_h1 else 0.0, 1.0 if is_ord_h2 else 0.0],
            'demand_h1': demand_h1, 'demand_h2': demand_h2,
            'inventory_h1': inv_h1_after, 'inventory_h2': inv_h2_after,
            'order_h1': order_h1, 'order_h2': order_h2,
            'actual_order_h1': actual_order_h1, 'actual_order_h2': actual_order_h2,
            'transferred_h1_to_h2': actual_trans_h1_to_h2, 'transferred_h2_to_h1': actual_trans_h2_to_h1,
            'shortage_h1': shortage_h1, 'shortage_h2': shortage_h2,
            'wastage_h1': wastage_h1, 'wastage_h2': wastage_h2,
            'total_cost': total_cost, 'wastage_cost': wastage_cost,
            'shortage_cost': shortage_cost, 'transshipment_cost': transshipment_cost,
            'order_cost': order_cost, 'inventory_holding_cost': inventory_holding_cost,
            'disruption_active_h1': mult_h1 > 1.0, 'disruption_active_h2': mult_h2 > 1.0,
            'crisis_counter_h1': self.crisis_counter_h1, 'crisis_counter_h2': self.crisis_counter_h2,
        }

        done = self.day >= self.episode_length
        return self._get_observation(), shared_reward, done, info

    def _process_order(self, hospital, amount):
        if amount <= 0: return 0.0
        start = hospital * self.shelf_life
        accepted = max(0.0, min(amount, self.max_capacity - np.sum(self.state[start:start + self.shelf_life])))
        self.state[start] += accepted
        return accepted

    def _transfer_blood(self, from_h, to_h, amount):
        if amount <= 0: return 0.0
        f_s, f_e = from_h * self.shelf_life, from_h * self.shelf_life + self.shelf_life
        t_s, t_e = to_h * self.shelf_life, to_h * self.shelf_life + self.shelf_life

        transferrable = max(0.0, min(amount, np.sum(self.state[f_s:f_e]), self.max_capacity - np.sum(self.state[t_s:t_e])))
        transferred = 0.0
        for i in range(f_e - 1, f_s - 1, -1):
            if self.state[i] > 0:
                to_transfer = min(self.state[i], transferrable - transferred)
                self.state[i] -= to_transfer
                self.state[t_s + (i - f_s)] += to_transfer
                transferred += to_transfer
                if transferred >= transferrable: break
        return transferred

    def _fulfill_demand(self, hospital, demand):
        s_idx, e_idx = hospital * self.shelf_life, hospital * self.shelf_life + self.shelf_life
        unfulfilled = demand
        for i in range(e_idx - 1, s_idx - 1, -1):
            if self.state[i] > 0 and unfulfilled > 0:
                used = min(self.state[i], unfulfilled)
                self.state[i] -= used
                unfulfilled -= used
            if unfulfilled <= 1e-5: break
        wastage = self.state[e_idx - 1]
        self.state[e_idx - 1] = 0
        return wastage, unfulfilled, demand - unfulfilled

    def _age_blood(self):
        for h in range(self.num_hospitals):
            s_idx, e_idx = h * self.shelf_life, h * self.shelf_life + self.shelf_life
            self.state[s_idx + 1:e_idx] = self.state[s_idx:e_idx - 1]
            self.state[s_idx] = 0

# =============================================================================
# RL ALGORITHMS (MAPPO)
# =============================================================================
class RunningMeanStd:
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = x.shape[0]
        self.update_from_moments(batch_mean, batch_var, batch_count)

    def update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count
        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count
        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def normalize(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


class Actor(nn.Module):
    def __init__(self, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(STATE_DIM, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.alpha_head = nn.Linear(256, action_dim)
        self.beta_head = nn.Linear(256, action_dim)

    def forward(self, state):
        feat = self.net(state)
        alpha = torch.clamp(F.softplus(self.alpha_head(feat)) + 1.0, 1.0, 100.0)
        beta = torch.clamp(F.softplus(self.beta_head(feat)) + 1.0, 1.0, 100.0)
        return Beta(alpha, beta)

    def get_action(self, state):
        with torch.no_grad():
            dist = self.forward(state)
            action = dist.sample()
            safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
            return safe_act.squeeze(0).cpu().numpy(), dist.log_prob(safe_act).squeeze(0).cpu().numpy()

    def evaluate(self, state, action):
        dist = self.forward(state)
        safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
        return dist.log_prob(safe_act), dist.entropy()

class MultiAgentMAPPO:
    def __init__(self):
        self.proc_actor = Actor(PROCUREMENT_ACTION_DIM).to(DEVICE)
        self.log_actor = Actor(LOGISTICS_ACTION_DIM).to(DEVICE)
        self.critic = nn.Sequential(nn.Linear(STATE_DIM, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1)).to(DEVICE)

        self.opt_p = Adam(self.proc_actor.parameters(), lr=LR)
        self.opt_l = Adam(self.log_actor.parameters(), lr=LR)
        self.opt_c = Adam(self.critic.parameters(), lr=LR)
        self.entropy_coef = ENTROPY_COEF
        self.rms = RunningMeanStd()

    def _norm(self, st):
        st = st.clone()
        st[:, :10] /= STATE_SCALE
        return st

    def get_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        ap, lp = self.proc_actor.get_action(st)
        al, ll = self.log_actor.get_action(st)
        return np.concatenate([ap, al]), ap, al, lp, ll

    def get_deterministic_joint_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        with torch.no_grad():
            act_proc = self.proc_actor.forward(st).mean.squeeze(0).cpu().numpy()
            act_log = self.log_actor.forward(st).mean.squeeze(0).cpu().numpy()
        return np.concatenate([act_proc, act_log], axis=-1)

    def save(self, path):
        torch.save({'proc_actor': self.proc_actor.state_dict(),
                    'log_actor': self.log_actor.state_dict(),
                    'critic': self.critic.state_dict()}, path)

    def load(self, path):
        ck = torch.load(path, map_location=DEVICE, weights_only=True)
        self.proc_actor.load_state_dict(ck['proc_actor'])
        self.log_actor.load_state_dict(ck['log_actor'])
        self.critic.load_state_dict(ck['critic'])

    def update(self, s, ap, al, lp, ll, r, ns, d, masks):
        s = self._norm(torch.FloatTensor(np.array(s)).to(DEVICE))
        ns = self._norm(torch.FloatTensor(np.array(ns)).to(DEVICE))
        ap = torch.FloatTensor(np.array(ap)).to(DEVICE)
        al = torch.FloatTensor(np.array(al)).to(DEVICE)
        lp = torch.FloatTensor(np.array(lp)).to(DEVICE)
        ll = torch.FloatTensor(np.array(ll)).to(DEVICE)

        r_np = np.array(r)
        self.rms.update(r_np)
        r_norm = torch.FloatTensor(self.rms.normalize(r_np)).to(DEVICE)

        d = torch.FloatTensor(np.array(d)).to(DEVICE)
        masks = torch.FloatTensor(np.array(masks)).to(DEVICE)

        with torch.no_grad():
            vals = self.critic(s).squeeze()
            n_vals = self.critic(ns).squeeze()

        adv = torch.zeros_like(r_norm).to(DEVICE)
        gae = 0
        for t in reversed(range(len(r_norm))):
            delta = r_norm[t] + GAMMA * n_vals[t] * (1 - d[t]) - vals[t]
            gae = delta + GAMMA * GAE_LAMBDA * (1 - d[t]) * gae
            adv[t] = gae

        returns = adv + vals
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(EPOCHS):
            new_logp_p, ent_p = self.proc_actor.evaluate(s, ap)
            ratio_p = torch.exp(new_logp_p - lp.detach())
            surr1_p = ratio_p * adv.unsqueeze(1)
            surr2_p = torch.clamp(ratio_p, 1-EPSILON, 1+EPSILON) * adv.unsqueeze(1)
            loss_p = -torch.min(surr1_p, surr2_p) * masks
            loss_p = loss_p.sum() / (masks.sum() + 1e-8)
            loss_p -= self.entropy_coef * (ent_p * masks).sum() / (masks.sum() + 1e-8)

            self.opt_p.zero_grad()
            loss_p.backward()
            nn.utils.clip_grad_norm_(self.proc_actor.parameters(), 0.5)
            self.opt_p.step()

            new_logp_l, ent_l = self.log_actor.evaluate(s, al)
            ratio_l = torch.exp(new_logp_l - ll.detach())
            surr1_l = ratio_l * adv.unsqueeze(1)
            surr2_l = torch.clamp(ratio_l, 1-EPSILON, 1+EPSILON) * adv.unsqueeze(1)
            loss_l = -torch.min(surr1_l, surr2_l).mean() - self.entropy_coef * ent_l.mean()

            self.opt_l.zero_grad()
            loss_l.backward()
            nn.utils.clip_grad_norm_(self.log_actor.parameters(), 0.5)
            self.opt_l.step()

            v_pred = self.critic(s).squeeze()
            loss_c = F.mse_loss(v_pred, returns)
            self.opt_c.zero_grad()
            loss_c.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
            self.opt_c.step()

        self.entropy_coef = max(MIN_ENTROPY, self.entropy_coef * ENTROPY_DECAY)

# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train(env_config, model_path, total_episodes=TOTAL_EPISODES):
    env = PlateletSupplyChainEnv(env_config)
    mappo = MultiAgentMAPPO()

    buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []

    for ep in tqdm(range(total_episodes), desc="Training"):
        s = env.reset()
        ep_r = 0
        for step in range(env.episode_length):
            j_act, ap, al, lp, ll = mappo.get_action(s)
            ns, r, d, info = env.step(j_act)

            buf_s.append(s); buf_ap.append(ap); buf_al.append(al)
            buf_lp.append(lp); buf_ll.append(ll); buf_r.append(r)
            buf_ns.append(ns); buf_d.append(d)
            buf_m.append(info['order_mask'])

            s = ns
            ep_r += r
            if d: break

        rewards_log.append(ep_r)

        if (ep + 1) % UPDATE_EPISODES == 0:
            mappo.update(buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_ap, buf_al, buf_lp, buf_ll, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], [], [], []

        if ep >= 100:
            avg_r = np.mean(rewards_log[-100:])
            if avg_r > best_reward:
                best_reward = avg_r
                mappo.save(model_path)

    return best_reward

# =============================================================================
# EVALUATION FUNCTION
# =============================================================================
def evaluate_multi_seed(env_config, model_path, seeds=range(1, 51)):
    env = PlateletSupplyChainEnv(env_config)
    mappo = MultiAgentMAPPO()
    try:
        mappo.load(model_path)
    except FileNotFoundError:
        print(f"Error: Model file '{model_path}' not found!")
        return None

    mappo.proc_actor.eval()
    mappo.log_actor.eval()

    all_seed_results = []

    for seed in tqdm(seeds, desc="Evaluating"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        state = env.reset()
        records = []

        for day in range(env.episode_length):
            action = mappo.get_deterministic_joint_action(state)
            next_state, reward, done, info = env.step(action)
            records.append(info)
            state = next_state
            if done: break

        df = pd.DataFrame(records)

        h1_shortage_cost = df['shortage_h1'] * env.shortage_cost
        h2_shortage_cost = df['shortage_h2'] * env.shortage_cost
        h1_wastage_cost = df['wastage_h1'] * env.wastage_cost
        h2_wastage_cost = df['wastage_h2'] * env.wastage_cost
        h1_order_cost = df['actual_order_h1'] * env.order_cost
        h2_order_cost = df['actual_order_h2'] * env.order_cost

        total_inv = df['inventory_h1'] + df['inventory_h2']
        h1_holding_cost = (df['inventory_h1'] / total_inv.replace(0, 1e-8)) * df['inventory_holding_cost']
        h2_holding_cost = df['inventory_holding_cost'] - h1_holding_cost

        h1_total_cost = h1_shortage_cost.sum() + h1_wastage_cost.sum() + h1_order_cost.sum() + h1_holding_cost.sum()
        h2_total_cost = h2_shortage_cost.sum() + h2_wastage_cost.sum() + h2_order_cost.sum() + h2_holding_cost.sum()
        total_trans_cost = df['transshipment_cost'].sum()
        network_total_cost = h1_total_cost + h2_total_cost + total_trans_cost

        h1_total_demand = df['demand_h1'].sum()
        h2_total_demand = df['demand_h2'].sum()
        h1_shortage_rate = (df['shortage_h1'].sum() / h1_total_demand * 100) if h1_total_demand > 0 else 0
        h2_shortage_rate = (df['shortage_h2'].sum() / h2_total_demand * 100) if h2_total_demand > 0 else 0
        h1_wastage_rate = (df['wastage_h1'].sum() / h1_total_demand * 100) if h1_total_demand > 0 else 0
        h2_wastage_rate = (df['wastage_h2'].sum() / h2_total_demand * 100) if h2_total_demand > 0 else 0

        total_trans_units = df['transferred_h1_to_h2'].sum() + df['transferred_h2_to_h1'].sum()

        seed_result = {
            'seed': seed,
            'h1_total_cost': h1_total_cost, 'h1_shortage_cost': h1_shortage_cost.sum(),
            'h1_shortage_rate': h1_shortage_rate, 'h1_wastage_cost': h1_wastage_cost.sum(),
            'h1_wastage_rate': h1_wastage_rate, 'h1_holding_cost': h1_holding_cost.sum(),
            'h1_order_cost': h1_order_cost.sum(),
            'h2_total_cost': h2_total_cost, 'h2_shortage_cost': h2_shortage_cost.sum(),
            'h2_shortage_rate': h2_shortage_rate, 'h2_wastage_cost': h2_wastage_cost.sum(),
            'h2_wastage_rate': h2_wastage_rate, 'h2_holding_cost': h2_holding_cost.sum(),
            'h2_order_cost': h2_order_cost.sum(),
            'network_total_cost': network_total_cost, 'total_trans_cost': total_trans_cost,
            'total_trans_units': total_trans_units,
            'trans_h1_to_h2': df['transferred_h1_to_h2'].sum(),
            'trans_h2_to_h1': df['transferred_h2_to_h1'].sum(),
        }
        all_seed_results.append(seed_result)

    return pd.DataFrame(all_seed_results)

# =============================================================================
# SENSITIVITY ANALYSIS DRIVER
# =============================================================================
def run_sensitivity_analysis():
    os.makedirs("sensitivity_results", exist_ok=True)

    # --- Analysis 1: disruption_mult sensitivity ---
    disruption_mult_values = [1.0, 1.7, 2]
    disruption_results = []

    print("\n" + "="*80)
    print("SENSITIVITY ANALYSIS 1: DISRUPTION MULTIPLIER (disruption_mult)")
    print("="*80)

    for dm in disruption_mult_values:
        print(f"\n>>> Training with disruption_mult = {dm}")
        env_cfg = BASE_ENV_CONFIG.copy()
        env_cfg["disruption_mult"] = dm

        model_path = f"sensitivity_results/mappo_dm_{dm:.1f}.pt"

        # Train from scratch
        best_reward = train(env_cfg, model_path, total_episodes=TOTAL_EPISODES)
        print(f"    Best 100-ep avg reward: {best_reward:.4f}")

        # Evaluate on 50 seeds
        print(f"    Evaluating on 50 seeds...")
        results_df = evaluate_multi_seed(env_cfg, model_path, seeds=range(1, 51))

        if results_df is not None:
            results_df.to_csv(f"sensitivity_results/results_dm_{dm:.1f}.csv", index=False)

            agg = {
                'disruption_mult': dm,
                'network_total_cost_mean': results_df['network_total_cost'].mean(),
                'network_total_cost_std': results_df['network_total_cost'].std(),
                'h1_shortage_rate_mean': results_df['h1_shortage_rate'].mean(),
                'h1_shortage_rate_std': results_df['h1_shortage_rate'].std(),
                'h2_shortage_rate_mean': results_df['h2_shortage_rate'].mean(),
                'h2_shortage_rate_std': results_df['h2_shortage_rate'].std(),
                'h1_wastage_rate_mean': results_df['h1_wastage_rate'].mean(),
                'h1_wastage_rate_std': results_df['h1_wastage_rate'].std(),
                'h2_wastage_rate_mean': results_df['h2_wastage_rate'].mean(),
                'h2_wastage_rate_std': results_df['h2_wastage_rate'].std(),
                'total_trans_cost_mean': results_df['total_trans_cost'].mean(),
                'total_trans_cost_std': results_df['total_trans_cost'].std(),
                'total_trans_units_mean': results_df['total_trans_units'].mean(),
                'total_trans_units_std': results_df['total_trans_units'].std(),
            }
            disruption_results.append(agg)

    # --- Analysis 2: wastage_cost / shortage_cost sensitivity ---
    cost_pairs = [(15, 15), (15, 45), (15, 60)]
    cost_results = []

    print("\n" + "="*80)
    print("SENSITIVITY ANALYSIS 2: WASTAGE_COST / SHORTAGE_COST RATIO")
    print("="*80)

    for wc, sc in cost_pairs:
        ratio = wc / sc
        print(f"\n>>> Training with wastage_cost={wc}, shortage_cost={sc} (ratio={ratio:.2f})")
        env_cfg = BASE_ENV_CONFIG.copy()
        env_cfg["wastage_cost"] = float(wc)
        env_cfg["shortage_cost"] = float(sc)

        model_path = f"sensitivity_results/mappo_wc{wc}_sc{sc}.pt"

        # Train from scratch
        best_reward = train(env_cfg, model_path, total_episodes=TOTAL_EPISODES)
        print(f"    Best 100-ep avg reward: {best_reward:.4f}")

        # Evaluate on 50 seeds
        print(f"    Evaluating on 50 seeds...")
        results_df = evaluate_multi_seed(env_cfg, model_path, seeds=range(1, 51))

        if results_df is not None:
            results_df.to_csv(f"sensitivity_results/results_wc{wc}_sc{sc}.csv", index=False)

            agg = {
                'wastage_cost': wc,
                'shortage_cost': sc,
                'ratio': ratio,
                'network_total_cost_mean': results_df['network_total_cost'].mean(),
                'network_total_cost_std': results_df['network_total_cost'].std(),
                'h1_shortage_rate_mean': results_df['h1_shortage_rate'].mean(),
                'h1_shortage_rate_std': results_df['h1_shortage_rate'].std(),
                'h2_shortage_rate_mean': results_df['h2_shortage_rate'].mean(),
                'h2_shortage_rate_std': results_df['h2_shortage_rate'].std(),
                'h1_wastage_rate_mean': results_df['h1_wastage_rate'].mean(),
                'h1_wastage_rate_std': results_df['h1_wastage_rate'].std(),
                'h2_wastage_rate_mean': results_df['h2_wastage_rate'].mean(),
                'h2_wastage_rate_std': results_df['h2_wastage_rate'].std(),
                'total_trans_cost_mean': results_df['total_trans_cost'].mean(),
                'total_trans_cost_std': results_df['total_trans_cost'].std(),
                'total_trans_units_mean': results_df['total_trans_units'].mean(),
                'total_trans_units_std': results_df['total_trans_units'].std(),
            }
            cost_results.append(agg)

    # =============================================================================
    # PRINT SUMMARY TABLES
    # =============================================================================
    print("\n\n" + "="*100)
    print("SUMMARY TABLE 1: SENSITIVITY TO DISRUPTION MULTIPLIER (disruption_mult)")
    print("="*100)

    header1 = (f"{'disruption_mult':>15} | {'Net_Cost':>12} | {'H1_Short%':>12} | {'H2_Short%':>12} | "
               f"{'H1_Waste%':>12} | {'H2_Waste%':>12} | {'Trans_Cost':>12} | {'Trans_Units':>12}")
    print(header1)
    print("-" * 100)

    for r in disruption_results:
        line = (f"{r['disruption_mult']:>15.1f} | "
                f"{r['network_total_cost_mean']:>10.2f}±{r['network_total_cost_std']:<4.2f} | "
                f"{r['h1_shortage_rate_mean']:>10.2f}±{r['h1_shortage_rate_std']:<4.2f} | "
                f"{r['h2_shortage_rate_mean']:>10.2f}±{r['h2_shortage_rate_std']:<4.2f} | "
                f"{r['h1_wastage_rate_mean']:>10.2f}±{r['h1_wastage_rate_std']:<4.2f} | "
                f"{r['h2_wastage_rate_mean']:>10.2f}±{r['h2_wastage_rate_std']:<4.2f} | "
                f"{r['total_trans_cost_mean']:>10.2f}±{r['total_trans_cost_std']:<4.2f} | "
                f"{r['total_trans_units_mean']:>10.2f}±{r['total_trans_units_std']:<4.2f}")
        print(line)

    print("\n" + "="*100)
    print("SUMMARY TABLE 2: SENSITIVITY TO WASTAGE_COST / SHORTAGE_COST RATIO")
    print("="*100)

    header2 = (f"{'WC/SC':>10} | {'Net_Cost':>12} | {'H1_Short%':>12} | {'H2_Short%':>12} | "
               f"{'H1_Waste%':>12} | {'H2_Waste%':>12} | {'Trans_Cost':>12} | {'Trans_Units':>12}")
    print(header2)
    print("-" * 100)

    for r in cost_results:
        label = f"{r['wastage_cost']:.0f}/{r['shortage_cost']:.0f}"
        line = (f"{label:>10} | "
                f"{r['network_total_cost_mean']:>10.2f}±{r['network_total_cost_std']:<4.2f} | "
                f"{r['h1_shortage_rate_mean']:>10.2f}±{r['h1_shortage_rate_std']:<4.2f} | "
                f"{r['h2_shortage_rate_mean']:>10.2f}±{r['h2_shortage_rate_std']:<4.2f} | "
                f"{r['h1_wastage_rate_mean']:>10.2f}±{r['h1_wastage_rate_std']:<4.2f} | "
                f"{r['h2_wastage_rate_mean']:>10.2f}±{r['h2_wastage_rate_std']:<4.2f} | "
                f"{r['total_trans_cost_mean']:>10.2f}±{r['total_trans_cost_std']:<4.2f} | "
                f"{r['total_trans_units_mean']:>10.2f}±{r['total_trans_units_std']:<4.2f}")
        print(line)

    # Save summary tables to CSV
    pd.DataFrame(disruption_results).to_csv("sensitivity_results/summary_disruption_mult.csv", index=False)
    pd.DataFrame(cost_results).to_csv("sensitivity_results/summary_cost_ratio.csv", index=False)
    print("\nSummary tables saved to sensitivity_results/")


if __name__ == "__main__":
    run_sensitivity_analysis()


SENSITIVITY ANALYSIS 1: DISRUPTION MULTIPLIER (disruption_mult)

>>> Training with disruption_mult = 1.0


Training: 100%|██████████| 2000/2000 [04:38<00:00,  7.19it/s]


    Best 100-ep avg reward: -48.2783
    Evaluating on 50 seeds...


Evaluating: 100%|██████████| 50/50 [00:03<00:00, 13.60it/s]



>>> Training with disruption_mult = 1.7


...


>>> Training with wastage_cost=15, shortage_cost=60 (ratio=0.25)


Training: 100%|██████████| 2000/2000 [04:31<00:00,  7.38it/s]


    Best 100-ep avg reward: -67.4820
    Evaluating on 50 seeds...


Evaluating: 100%|██████████| 50/50 [00:03<00:00, 14.00it/s]



SUMMARY TABLE 1: SENSITIVITY TO DISRUPTION MULTIPLIER (disruption_mult)
disruption_mult |     Net_Cost |    H1_Short% |    H2_Short% |    H1_Waste% |    H2_Waste% |   Trans_Cost |  Trans_Units
----------------------------------------------------------------------------------------------------
            1.0 |    4217.40±482.48 |       3.61±1.70 |       4.71±2.31 |       1.71±1.57 |       1.85±1.69 |     437.94±52.91 |     274.84±38.97
            1.7 |    4784.86±585.07 |       4.12±2.35 |       4.71±2.87 |       1.64±1.53 |       2.20±1.97 |     718.47±65.31 |     504.87±53.31
            2.0 |    5303.47±798.26 |       3.14±2.83 |       5.39±4.14 |       3.86±3.03 |       2.51±2.10 |     481.66±53.48 |     307.36±44.27

SUMMARY TABLE 2: SENSITIVITY TO WASTAGE_COST / SHORTAGE_COST RATIO
     WC/SC |     Net_Cost |    H1_Short% |    H2_Short% |    H1_Waste% |    H2_Waste% |   Trans_Cost |  Trans_Units
----------------------------------------------------------------------------------

# Signle Agent with Trasn

In [9]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta
from torch.optim import Adam
from collections import defaultdict, deque
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ZINB_H1_PARAMS = [
    {"phi": 0.25, "p": 0.48, "n": 15},
]

ZINB_H2_PARAMS = [
    {"phi": 0.25, "p": 0.48, "n": 15},
]

def _zinb_mean(params: dict) -> float:
    phi, p, n = params["phi"], params["p"], params["n"]
    return (1.0 - phi) * n * (1.0 - p) / p

def sample_zinb(phi: float, p: float, n: int) -> int:
    if np.random.random() < phi: return 0
    return int(np.random.negative_binomial(n, p))

def sample_hospital_demand(zinb_params: list, mult: float = 1.0) -> int:
    total = sum(sample_zinb(p["phi"], p["p"], p["n"]) for p in zinb_params)
    return max(0, int(total * mult))

# Base config - will be modified by sensitivity analysis
BASE_ENV_CONFIG = {
    "max_capacity": 100,
    "shelf_life": 5,
    "order_cost": 1.0,
    "transshipment_cost": 1.0,
    "transshipment_fixed_cost": 5.0,
    "shortage_cost": 25.0,
    "wastage_cost": 15.0,
    "inventory_cost": 0.5,
    "episode_length": 60,
    "order_cap": 70,
    "trans_cap": 50,
    "transshipment_deadband": 0.1,
    "disruption_threshold": 1.2,
    "disruption_prob": 0.02,
    "disruption_min_len": 3,
    "disruption_max_len": 8,
    "disruption_mult": 1.4,
}

ORDER_SCHEDULE_H1 = {1, 4, 6}
ORDER_SCHEDULE_H2 = {0, 3, 5}

STATE_DIM = 26
ACTION_DIM = 3          # [order_h1, order_h2, transshipment]
STATE_SCALE = 100.0
REWARD_SCALE = 200.0

# PPO Hyperparameters
LR = 3e-4
GAMMA = 0.99
EPSILON = 0.2
EPOCHS = 4
GAE_LAMBDA = 0.95
UPDATE_EPISODES = 4
ENTROPY_COEF = 0.05
ENTROPY_DECAY = 0.999
MIN_ENTROPY = 0.005
TOTAL_EPISODES = 2000

# =============================================================================
# ENVIRONMENT
# =============================================================================
class PlateletSupplyChainEnv:
    def __init__(self, env_config=None):
        if env_config is None:
            env_config = BASE_ENV_CONFIG
        for k, v in env_config.items():
            setattr(self, k, v)
        self.num_hospitals = 2
        self.demand_mean_h1 = sum(_zinb_mean(p) for p in ZINB_H1_PARAMS)
        self.demand_mean_h2 = sum(_zinb_mean(p) for p in ZINB_H2_PARAMS)
        self.reset()

    def _is_order_day_h1(self): return (self.day % 7) in ORDER_SCHEDULE_H1
    def _is_order_day_h2(self): return (self.day % 7) in ORDER_SCHEDULE_H2

    def _compute_disruption_flag(self, history):
        if len(history) < 5: return 0.0
        ma2 = np.mean(list(history)[-2:])
        ma30 = np.mean(list(history)[-30:])
        return 1.0 if (ma30 > 0 and ma2 > self.disruption_threshold * ma30) else 0.0

    def _get_observation(self):
        obs = np.zeros((STATE_DIM,), dtype=np.float32)
        idx = 0
        for h in range(self.num_hospitals):
            start = h * self.shelf_life
            for age in range(self.shelf_life):
                obs[idx] = self.state[start + age]
                idx += 1

        inv_h1 = np.sum(self.state[0:self.shelf_life])
        inv_h2 = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        obs[10] = self.day / self.episode_length
        obs[11] = self._compute_disruption_flag(self.demand_history_h1)
        obs[12] = self._compute_disruption_flag(self.demand_history_h2)
        obs[13] = inv_h1 / self.max_capacity
        obs[14] = inv_h2 / self.max_capacity
        obs[15] = 1.0 if self._is_order_day_h1() else 0.0
        obs[16] = 1.0 if self._is_order_day_h2() else 0.0
        obs[17] = (inv_h1 - inv_h2) / self.max_capacity
        obs[18] = self.crisis_counter_h1 / self.disruption_max_len
        obs[19] = self.crisis_counter_h2 / self.disruption_max_len

        d1 = np.mean(self.demand_history_h1[-7:]) if self.demand_history_h1 else self.demand_mean_h1
        d2 = np.mean(self.demand_history_h2[-7:]) if self.demand_history_h2 else self.demand_mean_h2
        obs[20] = min(inv_h1 / (d1 * 7 + 1e-8), 3.0)
        obs[21] = min(inv_h2 / (d2 * 7 + 1e-8), 3.0)

        if self.demand_history_h1:
            td1 = sum(self.demand_history_h1[-30:])
            obs[22] = sum(self.shortage_history_h1) / (td1 + 1e-8)
            obs[24] = sum(self.wastage_history_h1) / (td1 + 1e-8)
        if self.demand_history_h2:
            td2 = sum(self.demand_history_h2[-30:])
            obs[23] = sum(self.shortage_history_h2) / (td2 + 1e-8)
            obs[25] = sum(self.wastage_history_h2) / (td2 + 1e-8)
        return obs.copy()

    def reset(self):
        self.state = np.zeros((self.num_hospitals * self.shelf_life,), dtype=np.float32)
        self.day = 0
        self.demand_history_h1, self.demand_history_h2 = [], []
        self.shortage_history_h1, self.shortage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.wastage_history_h1, self.wastage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.disruption_counter_h1 = self.disruption_counter_h2 = 0
        self.crisis_counter_h1 = self.crisis_counter_h2 = 0
        return self._get_observation()

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        is_ord_h1 = self._is_order_day_h1()
        is_ord_h2 = self._is_order_day_h2()

        order_h1 = action[0] * self.order_cap if is_ord_h1 else 0.0
        order_h2 = action[1] * self.order_cap if is_ord_h2 else 0.0

        a_t = action[2] * 2.0 - 1.0
        tau = self.transshipment_deadband

        trans_h1_to_h2 = trans_h2_to_h1 = 0.0
        if a_t > tau:
            trans_h1_to_h2 = ((a_t - tau) / (1.0 - tau)) * self.trans_cap
        elif a_t < -tau:
            trans_h2_to_h1 = ((abs(a_t) - tau) / (1.0 - tau)) * self.trans_cap

        inv_h1_before = np.sum(self.state[0:self.shelf_life])
        inv_h2_before = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        actual_order_h1 = self._process_order(0, min(order_h1, max(0.0, self.max_capacity - inv_h1_before)))
        actual_order_h2 = self._process_order(1, min(order_h2, max(0.0, self.max_capacity - inv_h2_before)))

        actual_trans_h1_to_h2 = self._transfer_blood(0, 1, trans_h1_to_h2)
        actual_trans_h2_to_h1 = self._transfer_blood(1, 0, trans_h2_to_h1)

        if self.day >= 5:
            if self.disruption_counter_h1 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h1 = random.randint(self.disruption_min_len, self.disruption_max_len)
            if self.disruption_counter_h2 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h2 = random.randint(self.disruption_min_len, self.disruption_max_len)

        mult_h1 = self.disruption_mult if self.disruption_counter_h1 > 0 else 1.0
        mult_h2 = self.disruption_mult if self.disruption_counter_h2 > 0 else 1.0

        if self.disruption_counter_h1 > 0: self.disruption_counter_h1 -= 1
        if self.disruption_counter_h2 > 0: self.disruption_counter_h2 -= 1

        demand_h1 = sample_hospital_demand(ZINB_H1_PARAMS, mult=mult_h1)
        demand_h2 = sample_hospital_demand(ZINB_H2_PARAMS, mult=mult_h2)

        self.demand_history_h1.append(demand_h1)
        self.demand_history_h2.append(demand_h2)

        self.crisis_counter_h1 = min(self.crisis_counter_h1 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h1) > 0.5 else 0
        self.crisis_counter_h2 = min(self.crisis_counter_h2 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h2) > 0.5 else 0

        wastage_h1, shortage_h1, _ = self._fulfill_demand(0, demand_h1)
        wastage_h2, shortage_h2, _ = self._fulfill_demand(1, demand_h2)

        self.shortage_history_h1.append(shortage_h1)
        self.shortage_history_h2.append(shortage_h2)
        self.wastage_history_h1.append(wastage_h1)
        self.wastage_history_h2.append(wastage_h2)

        self._age_blood()

        inv_h1_after = np.sum(self.state[0:self.shelf_life])
        inv_h2_after = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        inventory_holding_cost = (inv_h1_before + inv_h2_before) * self.inventory_cost

        self.day += 1

        total_transferred = actual_trans_h1_to_h2 + actual_trans_h2_to_h1
        transshipment_cost = total_transferred * self.transshipment_cost + (self.transshipment_fixed_cost if total_transferred >= 1.0 else 0.0)
        wastage_cost = (wastage_h1 + wastage_h2) * self.wastage_cost
        shortage_cost = (shortage_h1 + shortage_h2) * self.shortage_cost
        order_cost = (actual_order_h1 + actual_order_h2) * self.order_cost

        total_cost = wastage_cost + shortage_cost + transshipment_cost + order_cost + inventory_holding_cost
        shared_reward = -total_cost / REWARD_SCALE

        info = {
            'order_mask': [1.0 if is_ord_h1 else 0.0, 1.0 if is_ord_h2 else 0.0],
            'demand_h1': demand_h1, 'demand_h2': demand_h2,
            'inventory_h1': inv_h1_after, 'inventory_h2': inv_h2_after,
            'order_h1': order_h1, 'order_h2': order_h2,
            'actual_order_h1': actual_order_h1, 'actual_order_h2': actual_order_h2,
            'transferred_h1_to_h2': actual_trans_h1_to_h2, 'transferred_h2_to_h1': actual_trans_h2_to_h1,
            'shortage_h1': shortage_h1, 'shortage_h2': shortage_h2,
            'wastage_h1': wastage_h1, 'wastage_h2': wastage_h2,
            'total_cost': total_cost, 'wastage_cost': wastage_cost,
            'shortage_cost': shortage_cost, 'transshipment_cost': transshipment_cost,
            'order_cost': order_cost, 'inventory_holding_cost': inventory_holding_cost,
            'disruption_active_h1': mult_h1 > 1.0, 'disruption_active_h2': mult_h2 > 1.0,
            'crisis_counter_h1': self.crisis_counter_h1, 'crisis_counter_h2': self.crisis_counter_h2,
        }

        done = self.day >= self.episode_length
        return self._get_observation(), shared_reward, done, info

    def _process_order(self, hospital, amount):
        if amount <= 0: return 0.0
        start = hospital * self.shelf_life
        accepted = max(0.0, min(amount, self.max_capacity - np.sum(self.state[start:start + self.shelf_life])))
        self.state[start] += accepted
        return accepted

    def _transfer_blood(self, from_h, to_h, amount):
        if amount <= 0: return 0.0
        f_s, f_e = from_h * self.shelf_life, from_h * self.shelf_life + self.shelf_life
        t_s, t_e = to_h * self.shelf_life, to_h * self.shelf_life + self.shelf_life

        transferrable = max(0.0, min(amount, np.sum(self.state[f_s:f_e]), self.max_capacity - np.sum(self.state[t_s:t_e])))
        transferred = 0.0
        for i in range(f_e - 1, f_s - 1, -1):
            if self.state[i] > 0:
                to_transfer = min(self.state[i], transferrable - transferred)
                self.state[i] -= to_transfer
                self.state[t_s + (i - f_s)] += to_transfer
                transferred += to_transfer
                if transferred >= transferrable: break
        return transferred

    def _fulfill_demand(self, hospital, demand):
        s_idx, e_idx = hospital * self.shelf_life, hospital * self.shelf_life + self.shelf_life
        unfulfilled = demand
        for i in range(e_idx - 1, s_idx - 1, -1):
            if self.state[i] > 0 and unfulfilled > 0:
                used = min(self.state[i], unfulfilled)
                self.state[i] -= used
                unfulfilled -= used
            if unfulfilled <= 1e-5: break
        wastage = self.state[e_idx - 1]
        self.state[e_idx - 1] = 0
        return wastage, unfulfilled, demand - unfulfilled

    def _age_blood(self):
        for h in range(self.num_hospitals):
            s_idx, e_idx = h * self.shelf_life, h * self.shelf_life + self.shelf_life
            self.state[s_idx + 1:e_idx] = self.state[s_idx:e_idx - 1]
            self.state[s_idx] = 0

# =============================================================================
# SINGLE AGENT PPO
# =============================================================================
class RunningMeanStd:
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = x.shape[0]
        self.update_from_moments(batch_mean, batch_var, batch_count)

    def update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count
        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count
        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def normalize(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


class Actor(nn.Module):
    def __init__(self, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(STATE_DIM, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.alpha_head = nn.Linear(256, action_dim)
        self.beta_head = nn.Linear(256, action_dim)

    def forward(self, state):
        feat = self.net(state)
        alpha = torch.clamp(F.softplus(self.alpha_head(feat)) + 1.0, 1.0, 100.0)
        beta = torch.clamp(F.softplus(self.beta_head(feat)) + 1.0, 1.0, 100.0)
        return Beta(alpha, beta)

    def get_action(self, state):
        with torch.no_grad():
            dist = self.forward(state)
            action = dist.sample()
            safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
            return safe_act.squeeze(0).cpu().numpy(), dist.log_prob(safe_act).squeeze(0).cpu().numpy()

    def evaluate(self, state, action):
        dist = self.forward(state)
        safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
        return dist.log_prob(safe_act), dist.entropy()


class SingleAgentPPO:
    def __init__(self):
        self.actor = Actor(ACTION_DIM).to(DEVICE)
        self.critic = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        ).to(DEVICE)

        self.opt_a = Adam(self.actor.parameters(), lr=LR)
        self.opt_c = Adam(self.critic.parameters(), lr=LR)
        self.entropy_coef = ENTROPY_COEF
        self.rms = RunningMeanStd()

    def _norm(self, st):
        st = st.clone()
        st[:, :10] /= STATE_SCALE
        return st

    def get_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        act, logp = self.actor.get_action(st)
        return act, logp

    def get_deterministic_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        with torch.no_grad():
            act = self.actor.forward(st).mean.squeeze(0).cpu().numpy()
        return act

    def save(self, path):
        torch.save({
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict()
        }, path)

    def load(self, path):
        ck = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ck['actor'])
        self.critic.load_state_dict(ck['critic'])

    def update(self, s, a, lp, r, ns, d, masks):
        s = self._norm(torch.FloatTensor(np.array(s)).to(DEVICE))
        ns = self._norm(torch.FloatTensor(np.array(ns)).to(DEVICE))
        a = torch.FloatTensor(np.array(a)).to(DEVICE)
        lp = torch.FloatTensor(np.array(lp)).to(DEVICE)

        r_np = np.array(r)
        self.rms.update(r_np)
        r_norm = torch.FloatTensor(self.rms.normalize(r_np)).to(DEVICE)

        d = torch.FloatTensor(np.array(d)).to(DEVICE)
        masks = torch.FloatTensor(np.array(masks)).to(DEVICE)

        with torch.no_grad():
            vals = self.critic(s).squeeze()
            n_vals = self.critic(ns).squeeze()

        adv = torch.zeros_like(r_norm).to(DEVICE)
        gae = 0
        for t in reversed(range(len(r_norm))):
            delta = r_norm[t] + GAMMA * n_vals[t] * (1 - d[t]) - vals[t]
            gae = delta + GAMMA * GAE_LAMBDA * (1 - d[t]) * gae
            adv[t] = gae

        returns = adv + vals
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(EPOCHS):
            new_logp, ent = self.actor.evaluate(s, a)

            ratio = torch.exp(new_logp - lp.detach())

            full_mask = torch.ones_like(ratio)
            full_mask[:, :2] = masks

            surr1 = ratio * adv.unsqueeze(1)
            surr2 = torch.clamp(ratio, 1 - EPSILON, 1 + EPSILON) * adv.unsqueeze(1)

            loss_a = -torch.min(surr1, surr2) * full_mask
            loss_a = loss_a.sum() / (full_mask.sum() + 1e-8)
            loss_a -= self.entropy_coef * (ent * full_mask).sum() / (full_mask.sum() + 1e-8)

            self.opt_a.zero_grad()
            loss_a.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
            self.opt_a.step()

            v_pred = self.critic(s).squeeze()
            loss_c = F.mse_loss(v_pred, returns)
            self.opt_c.zero_grad()
            loss_c.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
            self.opt_c.step()

        self.entropy_coef = max(MIN_ENTROPY, self.entropy_coef * ENTROPY_DECAY)

# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train(env_config, model_path, total_episodes=TOTAL_EPISODES):
    env = PlateletSupplyChainEnv(env_config)
    agent = SingleAgentPPO()

    buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []

    for ep in tqdm(range(total_episodes), desc="Training"):
        s = env.reset()
        ep_r = 0
        for step in range(env.episode_length):
            act, lp = agent.get_action(s)
            ns, r, d, info = env.step(act)

            buf_s.append(s)
            buf_a.append(act)
            buf_lp.append(lp)
            buf_r.append(r)
            buf_ns.append(ns)
            buf_d.append(d)
            buf_m.append(info['order_mask'])

            s = ns
            ep_r += r
            if d: break

        rewards_log.append(ep_r)

        if (ep + 1) % UPDATE_EPISODES == 0:
            agent.update(buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []

        if ep >= 100:
            avg_r = np.mean(rewards_log[-100:])
            if avg_r > best_reward:
                best_reward = avg_r
                agent.save(model_path)

    return best_reward

# =============================================================================
# EVALUATION FUNCTION
# =============================================================================
def evaluate_multi_seed(env_config, model_path, seeds=range(1, 51)):
    env = PlateletSupplyChainEnv(env_config)
    agent = SingleAgentPPO()
    try:
        agent.load(model_path)
    except FileNotFoundError:
        print(f"Error: Model file '{model_path}' not found!")
        return None

    agent.actor.eval()

    all_seed_results = []

    for seed in tqdm(seeds, desc="Evaluating"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        state = env.reset()
        records = []

        for day in range(env.episode_length):
            action = agent.get_deterministic_action(state)
            next_state, reward, done, info = env.step(action)
            records.append(info)
            state = next_state
            if done: break

        df = pd.DataFrame(records)

        h1_shortage_cost = df['shortage_h1'] * env.shortage_cost
        h2_shortage_cost = df['shortage_h2'] * env.shortage_cost
        h1_wastage_cost = df['wastage_h1'] * env.wastage_cost
        h2_wastage_cost = df['wastage_h2'] * env.wastage_cost
        h1_order_cost = df['actual_order_h1'] * env.order_cost
        h2_order_cost = df['actual_order_h2'] * env.order_cost

        total_inv = df['inventory_h1'] + df['inventory_h2']
        h1_holding_cost = (df['inventory_h1'] / total_inv.replace(0, 1e-8)) * df['inventory_holding_cost']
        h2_holding_cost = df['inventory_holding_cost'] - h1_holding_cost

        h1_total_cost = h1_shortage_cost.sum() + h1_wastage_cost.sum() + h1_order_cost.sum() + h1_holding_cost.sum()
        h2_total_cost = h2_shortage_cost.sum() + h2_wastage_cost.sum() + h2_order_cost.sum() + h2_holding_cost.sum()
        total_trans_cost = df['transshipment_cost'].sum()
        network_total_cost = h1_total_cost + h2_total_cost + total_trans_cost

        h1_total_demand = df['demand_h1'].sum()
        h2_total_demand = df['demand_h2'].sum()
        h1_shortage_rate = (df['shortage_h1'].sum() / h1_total_demand * 100) if h1_total_demand > 0 else 0
        h2_shortage_rate = (df['shortage_h2'].sum() / h2_total_demand * 100) if h2_total_demand > 0 else 0
        h1_wastage_rate = (df['wastage_h1'].sum() / h1_total_demand * 100) if h1_total_demand > 0 else 0
        h2_wastage_rate = (df['wastage_h2'].sum() / h2_total_demand * 100) if h2_total_demand > 0 else 0

        total_trans_units = df['transferred_h1_to_h2'].sum() + df['transferred_h2_to_h1'].sum()

        seed_result = {
            'seed': seed,
            'h1_total_cost': h1_total_cost, 'h1_shortage_cost': h1_shortage_cost.sum(),
            'h1_shortage_rate': h1_shortage_rate, 'h1_wastage_cost': h1_wastage_cost.sum(),
            'h1_wastage_rate': h1_wastage_rate, 'h1_holding_cost': h1_holding_cost.sum(),
            'h1_order_cost': h1_order_cost.sum(),
            'h2_total_cost': h2_total_cost, 'h2_shortage_cost': h2_shortage_cost.sum(),
            'h2_shortage_rate': h2_shortage_rate, 'h2_wastage_cost': h2_wastage_cost.sum(),
            'h2_wastage_rate': h2_wastage_rate, 'h2_holding_cost': h2_holding_cost.sum(),
            'h2_order_cost': h2_order_cost.sum(),
            'network_total_cost': network_total_cost, 'total_trans_cost': total_trans_cost,
            'total_trans_units': total_trans_units,
            'trans_h1_to_h2': df['transferred_h1_to_h2'].sum(),
            'trans_h2_to_h1': df['transferred_h2_to_h1'].sum(),
        }
        all_seed_results.append(seed_result)

    return pd.DataFrame(all_seed_results)

# =============================================================================
# SENSITIVITY ANALYSIS DRIVER
# =============================================================================
def run_sensitivity_analysis():
    os.makedirs("sensitivity_results_single", exist_ok=True)

    # --- Analysis 1: disruption_mult sensitivity ---
    disruption_mult_values = [1.0, 1.7, 2.0]
    disruption_results = []

    print("\n" + "="*80)
    print("SENSITIVITY ANALYSIS 1: DISRUPTION MULTIPLIER (Single-Agent PPO)")
    print("="*80)

    for dm in disruption_mult_values:
        print(f"\n>>> Training with disruption_mult = {dm}")
        env_cfg = BASE_ENV_CONFIG.copy()
        env_cfg["disruption_mult"] = dm

        model_path = f"sensitivity_results_single/single_ppo_dm_{dm:.1f}.pt"

        best_reward = train(env_cfg, model_path, total_episodes=TOTAL_EPISODES)
        print(f"    Best 100-ep avg reward: {best_reward:.4f}")

        print(f"    Evaluating on 50 seeds...")
        results_df = evaluate_multi_seed(env_cfg, model_path, seeds=range(1, 51))

        if results_df is not None:
            results_df.to_csv(f"sensitivity_results_single/results_dm_{dm:.1f}.csv", index=False)

            agg = {
                'disruption_mult': dm,
                'network_total_cost_mean': results_df['network_total_cost'].mean(),
                'network_total_cost_std': results_df['network_total_cost'].std(),
                'h1_shortage_rate_mean': results_df['h1_shortage_rate'].mean(),
                'h1_shortage_rate_std': results_df['h1_shortage_rate'].std(),
                'h2_shortage_rate_mean': results_df['h2_shortage_rate'].mean(),
                'h2_shortage_rate_std': results_df['h2_shortage_rate'].std(),
                'h1_wastage_rate_mean': results_df['h1_wastage_rate'].mean(),
                'h1_wastage_rate_std': results_df['h1_wastage_rate'].std(),
                'h2_wastage_rate_mean': results_df['h2_wastage_rate'].mean(),
                'h2_wastage_rate_std': results_df['h2_wastage_rate'].std(),
                'total_trans_cost_mean': results_df['total_trans_cost'].mean(),
                'total_trans_cost_std': results_df['total_trans_cost'].std(),
                'total_trans_units_mean': results_df['total_trans_units'].mean(),
                'total_trans_units_std': results_df['total_trans_units'].std(),
            }
            disruption_results.append(agg)

    # --- Analysis 2: wastage_cost / shortage_cost sensitivity ---
    cost_pairs = [(15, 15), (15, 45), (15, 60)]
    cost_results = []

    print("\n" + "="*80)
    print("SENSITIVITY ANALYSIS 2: WASTAGE_COST / SHORTAGE_COST RATIO (Single-Agent PPO)")
    print("="*80)

    for wc, sc in cost_pairs:
        ratio = wc / sc
        print(f"\n>>> Training with wastage_cost={wc}, shortage_cost={sc} (ratio={ratio:.2f})")
        env_cfg = BASE_ENV_CONFIG.copy()
        env_cfg["wastage_cost"] = float(wc)
        env_cfg["shortage_cost"] = float(sc)

        model_path = f"sensitivity_results_single/single_ppo_wc{wc}_sc{sc}.pt"

        best_reward = train(env_cfg, model_path, total_episodes=TOTAL_EPISODES)
        print(f"    Best 100-ep avg reward: {best_reward:.4f}")

        print(f"    Evaluating on 50 seeds...")
        results_df = evaluate_multi_seed(env_cfg, model_path, seeds=range(1, 51))

        if results_df is not None:
            results_df.to_csv(f"sensitivity_results_single/results_wc{wc}_sc{sc}.csv", index=False)

            agg = {
                'wastage_cost': wc,
                'shortage_cost': sc,
                'ratio': ratio,
                'network_total_cost_mean': results_df['network_total_cost'].mean(),
                'network_total_cost_std': results_df['network_total_cost'].std(),
                'h1_shortage_rate_mean': results_df['h1_shortage_rate'].mean(),
                'h1_shortage_rate_std': results_df['h1_shortage_rate'].std(),
                'h2_shortage_rate_mean': results_df['h2_shortage_rate'].mean(),
                'h2_shortage_rate_std': results_df['h2_shortage_rate'].std(),
                'h1_wastage_rate_mean': results_df['h1_wastage_rate'].mean(),
                'h1_wastage_rate_std': results_df['h1_wastage_rate'].std(),
                'h2_wastage_rate_mean': results_df['h2_wastage_rate'].mean(),
                'h2_wastage_rate_std': results_df['h2_wastage_rate'].std(),
                'total_trans_cost_mean': results_df['total_trans_cost'].mean(),
                'total_trans_cost_std': results_df['total_trans_cost'].std(),
                'total_trans_units_mean': results_df['total_trans_units'].mean(),
                'total_trans_units_std': results_df['total_trans_units'].std(),
            }
            cost_results.append(agg)

    # =============================================================================
    # PRINT SUMMARY TABLES
    # =============================================================================
    print("\n\n" + "="*100)
    print("SUMMARY TABLE 1: SENSITIVITY TO DISRUPTION MULTIPLIER (Single-Agent PPO)")
    print("="*100)

    header1 = (f"{'disruption_mult':>15} | {'Net_Cost':>12} | {'H1_Short%':>12} | {'H2_Short%':>12} | "
               f"{'H1_Waste%':>12} | {'H2_Waste%':>12} | {'Trans_Cost':>12} | {'Trans_Units':>12}")
    print(header1)
    print("-" * 100)

    for r in disruption_results:
        line = (f"{r['disruption_mult']:>15.1f} | "
                f"{r['network_total_cost_mean']:>10.2f}±{r['network_total_cost_std']:<4.2f} | "
                f"{r['h1_shortage_rate_mean']:>10.2f}±{r['h1_shortage_rate_std']:<4.2f} | "
                f"{r['h2_shortage_rate_mean']:>10.2f}±{r['h2_shortage_rate_std']:<4.2f} | "
                f"{r['h1_wastage_rate_mean']:>10.2f}±{r['h1_wastage_rate_std']:<4.2f} | "
                f"{r['h2_wastage_rate_mean']:>10.2f}±{r['h2_wastage_rate_std']:<4.2f} | "
                f"{r['total_trans_cost_mean']:>10.2f}±{r['total_trans_cost_std']:<4.2f} | "
                f"{r['total_trans_units_mean']:>10.2f}±{r['total_trans_units_std']:<4.2f}")
        print(line)

    print("\n" + "="*100)
    print("SUMMARY TABLE 2: SENSITIVITY TO WASTAGE_COST / SHORTAGE_COST RATIO (Single-Agent PPO)")
    print("="*100)

    header2 = (f"{'WC/SC':>10} | {'Net_Cost':>12} | {'H1_Short%':>12} | {'H2_Short%':>12} | "
               f"{'H1_Waste%':>12} | {'H2_Waste%':>12} | {'Trans_Cost':>12} | {'Trans_Units':>12}")
    print(header2)
    print("-" * 100)

    for r in cost_results:
        label = f"{r['wastage_cost']:.0f}/{r['shortage_cost']:.0f}"
        line = (f"{label:>10} | "
                f"{r['network_total_cost_mean']:>10.2f}±{r['network_total_cost_std']:<4.2f} | "
                f"{r['h1_shortage_rate_mean']:>10.2f}±{r['h1_shortage_rate_std']:<4.2f} | "
                f"{r['h2_shortage_rate_mean']:>10.2f}±{r['h2_shortage_rate_std']:<4.2f} | "
                f"{r['h1_wastage_rate_mean']:>10.2f}±{r['h1_wastage_rate_std']:<4.2f} | "
                f"{r['h2_wastage_rate_mean']:>10.2f}±{r['h2_wastage_rate_std']:<4.2f} | "
                f"{r['total_trans_cost_mean']:>10.2f}±{r['total_trans_cost_std']:<4.2f} | "
                f"{r['total_trans_units_mean']:>10.2f}±{r['total_trans_units_std']:<4.2f}")
        print(line)

    pd.DataFrame(disruption_results).to_csv("sensitivity_results_single/summary_disruption_mult.csv", index=False)
    pd.DataFrame(cost_results).to_csv("sensitivity_results_single/summary_cost_ratio.csv", index=False)
    print("\nSummary tables saved to sensitivity_results_single/")


if __name__ == "__main__":
    run_sensitivity_analysis()


SENSITIVITY ANALYSIS 1: DISRUPTION MULTIPLIER (Single-Agent PPO)

>>> Training with disruption_mult = 1.0


Training: 100%|██████████| 2000/2000 [02:47<00:00, 11.96it/s]


    Best 100-ep avg reward: -27.0503
    Evaluating on 50 seeds...


Evaluating: 100%|██████████| 50/50 [00:02<00:00, 20.72it/s]



>>> Training with disruption_mult = 1.7


...


>>> Training with wastage_cost=15, shortage_cost=60 (ratio=0.25)


Training: 100%|██████████| 2000/2000 [02:47<00:00, 11.94it/s]


    Best 100-ep avg reward: -33.7266
    Evaluating on 50 seeds...


Evaluating: 100%|██████████| 50/50 [00:02<00:00, 19.98it/s]



SUMMARY TABLE 1: SENSITIVITY TO DISRUPTION MULTIPLIER (Single-Agent PPO)
disruption_mult |     Net_Cost |    H1_Short% |    H2_Short% |    H1_Waste% |    H2_Waste% |   Trans_Cost |  Trans_Units
----------------------------------------------------------------------------------------------------
            1.0 |    4639.38±536.71 |       2.17±1.55 |       0.75±0.81 |       2.13±1.65 |       5.09±3.40 |     459.79±41.04 |     291.19±30.57
            1.7 |    5383.83±777.18 |       3.13±2.04 |       2.25±2.23 |       1.50±1.44 |       5.21±3.71 |     621.87±58.97 |     421.37±49.97
            2.0 |    6416.92±876.12 |       1.12±1.37 |       3.49±3.15 |       9.54±4.58 |       2.33±1.91 |     588.50±65.75 |     404.20±56.65

SUMMARY TABLE 2: SENSITIVITY TO WASTAGE_COST / SHORTAGE_COST RATIO (Single-Agent PPO)
     WC/SC |     Net_Cost |    H1_Short% |    H2_Short% |    H1_Waste% |    H2_Waste% |   Trans_Cost |  Trans_Units
--------------------------------------------------------------

# Single Agent without Trans

In [10]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta
from torch.optim import Adam
from collections import defaultdict, deque
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION & HYPERPARAMETERS
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ZINB_H1_PARAMS = [
    {"phi": 0.25, "p": 0.48, "n": 15},
]

ZINB_H2_PARAMS = [
    {"phi": 0.25, "p": 0.48, "n": 15},
]

def _zinb_mean(params: dict) -> float:
    phi, p, n = params["phi"], params["p"], params["n"]
    return (1.0 - phi) * n * (1.0 - p) / p

def sample_zinb(phi: float, p: float, n: int) -> int:
    if np.random.random() < phi: return 0
    return int(np.random.negative_binomial(n, p))

def sample_hospital_demand(zinb_params: list, mult: float = 1.0) -> int:
    total = sum(sample_zinb(p["phi"], p["p"], p["n"]) for p in zinb_params)
    return max(0, int(total * mult))

# Base config - will be modified by sensitivity analysis
BASE_ENV_CONFIG = {
    "max_capacity": 100,
    "shelf_life": 5,
    "order_cost": 1.0,
    "transshipment_cost": 1.0,
    "transshipment_fixed_cost": 5.0,
    "shortage_cost": 25.0,
    "wastage_cost": 15.0,
    "inventory_cost": 0.5,
    "episode_length": 60,
    "order_cap": 70,
    "trans_cap": 50,
    "transshipment_deadband": 0.1,
    "disruption_threshold": 1.2,
    "disruption_prob": 0.02,
    "disruption_min_len": 3,
    "disruption_max_len": 8,
    "disruption_mult": 1.4,
}

ORDER_SCHEDULE_H1 = {1, 4, 6}
ORDER_SCHEDULE_H2 = {0, 3, 5}

STATE_DIM = 26
ACTION_DIM = 2          # Only procurement [order_h1, order_h2]
STATE_SCALE = 70.0
REWARD_SCALE = 100.0

# PPO Hyperparameters
LR = 3e-4
GAMMA = 0.99
EPSILON = 0.2
EPOCHS = 4
GAE_LAMBDA = 0.95
UPDATE_EPISODES = 4
ENTROPY_COEF = 0.05
ENTROPY_DECAY = 0.999
MIN_ENTROPY = 0.005
TOTAL_EPISODES = 2000

# =============================================================================
# ENVIRONMENT (No Transshipment)
# =============================================================================
class PlateletSupplyChainEnv:
    def __init__(self, env_config=None):
        if env_config is None:
            env_config = BASE_ENV_CONFIG
        for k, v in env_config.items():
            setattr(self, k, v)
        self.num_hospitals = 2
        self.demand_mean_h1 = sum(_zinb_mean(p) for p in ZINB_H1_PARAMS)
        self.demand_mean_h2 = sum(_zinb_mean(p) for p in ZINB_H2_PARAMS)
        self.reset()

    def _is_order_day_h1(self): return (self.day % 7) in ORDER_SCHEDULE_H1
    def _is_order_day_h2(self): return (self.day % 7) in ORDER_SCHEDULE_H2

    def _compute_disruption_flag(self, history):
        if len(history) < 5: return 0.0
        ma2 = np.mean(list(history)[-2:])
        ma30 = np.mean(list(history)[-30:])
        return 1.0 if (ma30 > 0 and ma2 > self.disruption_threshold * ma30) else 0.0

    def _get_observation(self):
        obs = np.zeros((STATE_DIM,), dtype=np.float32)
        idx = 0
        for h in range(self.num_hospitals):
            start = h * self.shelf_life
            for age in range(self.shelf_life):
                obs[idx] = self.state[start + age]
                idx += 1

        inv_h1 = np.sum(self.state[0:self.shelf_life])
        inv_h2 = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        obs[10] = self.day / self.episode_length
        obs[11] = self._compute_disruption_flag(self.demand_history_h1)
        obs[12] = self._compute_disruption_flag(self.demand_history_h2)
        obs[13] = inv_h1 / self.max_capacity
        obs[14] = inv_h2 / self.max_capacity
        obs[15] = 1.0 if self._is_order_day_h1() else 0.0
        obs[16] = 1.0 if self._is_order_day_h2() else 0.0
        obs[17] = (inv_h1 - inv_h2) / self.max_capacity
        obs[18] = self.crisis_counter_h1 / self.disruption_max_len
        obs[19] = self.crisis_counter_h2 / self.disruption_max_len

        d1 = np.mean(self.demand_history_h1[-7:]) if self.demand_history_h1 else self.demand_mean_h1
        d2 = np.mean(self.demand_history_h2[-7:]) if self.demand_history_h2 else self.demand_mean_h2
        obs[20] = min(inv_h1 / (d1 * 7 + 1e-8), 3.0)
        obs[21] = min(inv_h2 / (d2 * 7 + 1e-8), 3.0)

        if self.demand_history_h1:
            td1 = sum(self.demand_history_h1[-30:])
            obs[22] = sum(self.shortage_history_h1) / (td1 + 1e-8)
            obs[24] = sum(self.wastage_history_h1) / (td1 + 1e-8)
        if self.demand_history_h2:
            td2 = sum(self.demand_history_h2[-30:])
            obs[23] = sum(self.shortage_history_h2) / (td2 + 1e-8)
            obs[25] = sum(self.wastage_history_h2) / (td2 + 1e-8)
        return obs.copy()

    def reset(self):
        self.state = np.zeros((self.num_hospitals * self.shelf_life,), dtype=np.float32)
        self.day = 0
        self.demand_history_h1, self.demand_history_h2 = [], []
        self.shortage_history_h1, self.shortage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.wastage_history_h1, self.wastage_history_h2 = deque(maxlen=30), deque(maxlen=30)
        self.disruption_counter_h1 = self.disruption_counter_h2 = 0
        self.crisis_counter_h1 = self.crisis_counter_h2 = 0
        return self._get_observation()

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        is_ord_h1 = self._is_order_day_h1()
        is_ord_h2 = self._is_order_day_h2()

        order_h1 = action[0] * self.order_cap if is_ord_h1 else 0.0
        order_h2 = action[1] * self.order_cap if is_ord_h2 else 0.0

        inv_h1_before = np.sum(self.state[0:self.shelf_life])
        inv_h2_before = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        actual_order_h1 = self._process_order(0, min(order_h1, max(0.0, self.max_capacity - inv_h1_before)))
        actual_order_h2 = self._process_order(1, min(order_h2, max(0.0, self.max_capacity - inv_h2_before)))

        # No transshipment
        actual_trans_h1_to_h2 = 0.0
        actual_trans_h2_to_h1 = 0.0

        if self.day >= 5:
            if self.disruption_counter_h1 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h1 = random.randint(self.disruption_min_len, self.disruption_max_len)
            if self.disruption_counter_h2 == 0 and random.random() < self.disruption_prob:
                self.disruption_counter_h2 = random.randint(self.disruption_min_len, self.disruption_max_len)

        mult_h1 = self.disruption_mult if self.disruption_counter_h1 > 0 else 1.0
        mult_h2 = self.disruption_mult if self.disruption_counter_h2 > 0 else 1.0

        if self.disruption_counter_h1 > 0: self.disruption_counter_h1 -= 1
        if self.disruption_counter_h2 > 0: self.disruption_counter_h2 -= 1

        demand_h1 = sample_hospital_demand(ZINB_H1_PARAMS, mult=mult_h1)
        demand_h2 = sample_hospital_demand(ZINB_H2_PARAMS, mult=mult_h2)

        self.demand_history_h1.append(demand_h1)
        self.demand_history_h2.append(demand_h2)

        self.crisis_counter_h1 = min(self.crisis_counter_h1 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h1) > 0.5 else 0
        self.crisis_counter_h2 = min(self.crisis_counter_h2 + 1, self.disruption_max_len) if self._compute_disruption_flag(self.demand_history_h2) > 0.5 else 0

        wastage_h1, shortage_h1, _ = self._fulfill_demand(0, demand_h1)
        wastage_h2, shortage_h2, _ = self._fulfill_demand(1, demand_h2)

        self.shortage_history_h1.append(shortage_h1)
        self.shortage_history_h2.append(shortage_h2)
        self.wastage_history_h1.append(wastage_h1)
        self.wastage_history_h2.append(wastage_h2)

        self._age_blood()

        inv_h1_after = np.sum(self.state[0:self.shelf_life])
        inv_h2_after = np.sum(self.state[self.shelf_life:self.shelf_life*2])

        inventory_holding_cost = (inv_h1_before + inv_h2_before) * self.inventory_cost
        self.day += 1

        # No transshipment cost
        transshipment_cost = 0.0
        wastage_cost = (wastage_h1 + wastage_h2) * self.wastage_cost
        shortage_cost = (shortage_h1 + shortage_h2) * self.shortage_cost
        order_cost = (actual_order_h1 + actual_order_h2) * self.order_cost

        total_cost = wastage_cost + shortage_cost + order_cost + inventory_holding_cost
        shared_reward = -total_cost / REWARD_SCALE

        info = {
            'order_mask': [1.0 if is_ord_h1 else 0.0, 1.0 if is_ord_h2 else 0.0],
            'demand_h1': demand_h1, 'demand_h2': demand_h2,
            'inventory_h1': inv_h1_after, 'inventory_h2': inv_h2_after,
            'order_h1': order_h1, 'order_h2': order_h2,
            'actual_order_h1': actual_order_h1, 'actual_order_h2': actual_order_h2,
            'transferred_h1_to_h2': actual_trans_h1_to_h2, 'transferred_h2_to_h1': actual_trans_h2_to_h1,
            'shortage_h1': shortage_h1, 'shortage_h2': shortage_h2,
            'wastage_h1': wastage_h1, 'wastage_h2': wastage_h2,
            'total_cost': total_cost, 'wastage_cost': wastage_cost,
            'shortage_cost': shortage_cost, 'transshipment_cost': transshipment_cost,
            'order_cost': order_cost, 'inventory_holding_cost': inventory_holding_cost,
            'disruption_active_h1': mult_h1 > 1.0, 'disruption_active_h2': mult_h2 > 1.0,
            'crisis_counter_h1': self.crisis_counter_h1, 'crisis_counter_h2': self.crisis_counter_h2,
        }

        done = self.day >= self.episode_length
        return self._get_observation(), shared_reward, done, info

    def _process_order(self, hospital, amount):
        if amount <= 0: return 0.0
        start = hospital * self.shelf_life
        accepted = max(0.0, min(amount, self.max_capacity - np.sum(self.state[start:start + self.shelf_life])))
        self.state[start] += accepted
        return accepted

    def _fulfill_demand(self, hospital, demand):
        s_idx, e_idx = hospital * self.shelf_life, hospital * self.shelf_life + self.shelf_life
        unfulfilled = demand
        for i in range(e_idx - 1, s_idx - 1, -1):
            if self.state[i] > 0 and unfulfilled > 0:
                used = min(self.state[i], unfulfilled)
                self.state[i] -= used
                unfulfilled -= used
            if unfulfilled <= 1e-5: break
        wastage = self.state[e_idx - 1]
        self.state[e_idx - 1] = 0
        return wastage, unfulfilled, demand - unfulfilled

    def _age_blood(self):
        for h in range(self.num_hospitals):
            s_idx, e_idx = h * self.shelf_life, h * self.shelf_life + self.shelf_life
            self.state[s_idx + 1:e_idx] = self.state[s_idx:e_idx - 1]
            self.state[s_idx] = 0

# =============================================================================
# SINGLE AGENT PPO (No Transshipment)
# =============================================================================
class RunningMeanStd:
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = x.shape[0]
        self.update_from_moments(batch_mean, batch_var, batch_count)

    def update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count
        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + delta**2 * self.count * batch_count / tot_count
        new_var = M2 / tot_count
        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def normalize(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


class Actor(nn.Module):
    def __init__(self, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(STATE_DIM, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.alpha_head = nn.Linear(256, action_dim)
        self.beta_head = nn.Linear(256, action_dim)

    def forward(self, state):
        feat = self.net(state)
        alpha = torch.clamp(F.softplus(self.alpha_head(feat)) + 1.0, 1.0, 100.0)
        beta = torch.clamp(F.softplus(self.beta_head(feat)) + 1.0, 1.0, 100.0)
        return Beta(alpha, beta)

    def get_action(self, state):
        with torch.no_grad():
            dist = self.forward(state)
            action = dist.sample()
            safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
            return safe_act.squeeze(0).cpu().numpy(), dist.log_prob(safe_act).squeeze(0).cpu().numpy()

    def evaluate(self, state, action):
        dist = self.forward(state)
        safe_act = torch.clamp(action, 1e-5, 1.0 - 1e-5)
        return dist.log_prob(safe_act), dist.entropy()


class SingleAgentPPO:
    def __init__(self):
        self.actor = Actor(ACTION_DIM).to(DEVICE)
        self.critic = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        ).to(DEVICE)

        self.opt_a = Adam(self.actor.parameters(), lr=LR)
        self.opt_c = Adam(self.critic.parameters(), lr=LR)
        self.entropy_coef = ENTROPY_COEF
        self.rms = RunningMeanStd()

    def _norm(self, st):
        st = st.clone()
        st[:, :10] /= STATE_SCALE
        return st

    def get_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        act, logp = self.actor.get_action(st)
        return act, logp

    def get_deterministic_action(self, state):
        st = self._norm(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
        with torch.no_grad():
            act = self.actor.forward(st).mean.squeeze(0).cpu().numpy()
        return act

    def save(self, path):
        torch.save({
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict()
        }, path)

    def load(self, path):
        ck = torch.load(path, map_location=DEVICE, weights_only=True)
        self.actor.load_state_dict(ck['actor'])
        self.critic.load_state_dict(ck['critic'])

    def update(self, s, a, lp, r, ns, d, masks):
        s = self._norm(torch.FloatTensor(np.array(s)).to(DEVICE))
        ns = self._norm(torch.FloatTensor(np.array(ns)).to(DEVICE))
        a = torch.FloatTensor(np.array(a)).to(DEVICE)
        lp = torch.FloatTensor(np.array(lp)).to(DEVICE)

        r_np = np.array(r)
        self.rms.update(r_np)
        r_norm = torch.FloatTensor(self.rms.normalize(r_np)).to(DEVICE)

        d = torch.FloatTensor(np.array(d)).to(DEVICE)
        masks = torch.FloatTensor(np.array(masks)).to(DEVICE)

        with torch.no_grad():
            vals = self.critic(s).squeeze()
            n_vals = self.critic(ns).squeeze()

        adv = torch.zeros_like(r_norm).to(DEVICE)
        gae = 0
        for t in reversed(range(len(r_norm))):
            delta = r_norm[t] + GAMMA * n_vals[t] * (1 - d[t]) - vals[t]
            gae = delta + GAMMA * GAE_LAMBDA * (1 - d[t]) * gae
            adv[t] = gae

        returns = adv + vals
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        for _ in range(EPOCHS):
            new_logp, ent = self.actor.evaluate(s, a)

            ratio = torch.exp(new_logp - lp.detach())
            surr1 = ratio * adv.unsqueeze(1)
            surr2 = torch.clamp(ratio, 1 - EPSILON, 1 + EPSILON) * adv.unsqueeze(1)

            # Both actions are procurement, so mask applies to all
            loss_a = -torch.min(surr1, surr2) * masks
            loss_a = loss_a.sum() / (masks.sum() + 1e-8)
            loss_a -= self.entropy_coef * (ent * masks).sum() / (masks.sum() + 1e-8)

            self.opt_a.zero_grad()
            loss_a.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), 0.5)
            self.opt_a.step()

            v_pred = self.critic(s).squeeze()
            loss_c = F.mse_loss(v_pred, returns)
            self.opt_c.zero_grad()
            loss_c.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), 0.5)
            self.opt_c.step()

        self.entropy_coef = max(MIN_ENTROPY, self.entropy_coef * ENTROPY_DECAY)

# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train(env_config, model_path, total_episodes=TOTAL_EPISODES):
    env = PlateletSupplyChainEnv(env_config)
    agent = SingleAgentPPO()

    buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []
    best_reward = -float('inf')
    rewards_log = []

    for ep in tqdm(range(total_episodes), desc="Training"):
        s = env.reset()
        ep_r = 0
        for step in range(env.episode_length):
            act, lp = agent.get_action(s)
            ns, r, d, info = env.step(act)

            buf_s.append(s)
            buf_a.append(act)
            buf_lp.append(lp)
            buf_r.append(r)
            buf_ns.append(ns)
            buf_d.append(d)
            buf_m.append(info['order_mask'])

            s = ns
            ep_r += r
            if d: break

        rewards_log.append(ep_r)

        if (ep + 1) % UPDATE_EPISODES == 0:
            agent.update(buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m)
            buf_s, buf_a, buf_lp, buf_r, buf_ns, buf_d, buf_m = [], [], [], [], [], [], []

        if ep >= 100:
            avg_r = np.mean(rewards_log[-100:])
            if avg_r > best_reward:
                best_reward = avg_r
                agent.save(model_path)

    return best_reward

# =============================================================================
# EVALUATION FUNCTION
# =============================================================================
def evaluate_multi_seed(env_config, model_path, seeds=range(1, 51)):
    env = PlateletSupplyChainEnv(env_config)
    agent = SingleAgentPPO()
    try:
        agent.load(model_path)
    except FileNotFoundError:
        print(f"Error: Model file '{model_path}' not found!")
        return None

    agent.actor.eval()

    all_seed_results = []

    for seed in tqdm(seeds, desc="Evaluating"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        state = env.reset()
        records = []

        for day in range(env.episode_length):
            action = agent.get_deterministic_action(state)
            next_state, reward, done, info = env.step(action)
            records.append(info)
            state = next_state
            if done: break

        df = pd.DataFrame(records)

        h1_shortage_cost = df['shortage_h1'] * env.shortage_cost
        h2_shortage_cost = df['shortage_h2'] * env.shortage_cost
        h1_wastage_cost = df['wastage_h1'] * env.wastage_cost
        h2_wastage_cost = df['wastage_h2'] * env.wastage_cost
        h1_order_cost = df['actual_order_h1'] * env.order_cost
        h2_order_cost = df['actual_order_h2'] * env.order_cost

        total_inv = df['inventory_h1'] + df['inventory_h2']
        h1_holding_cost = (df['inventory_h1'] / total_inv.replace(0, 1e-8)) * df['inventory_holding_cost']
        h2_holding_cost = df['inventory_holding_cost'] - h1_holding_cost

        h1_total_cost = h1_shortage_cost.sum() + h1_wastage_cost.sum() + h1_order_cost.sum() + h1_holding_cost.sum()
        h2_total_cost = h2_shortage_cost.sum() + h2_wastage_cost.sum() + h2_order_cost.sum() + h2_holding_cost.sum()
        total_trans_cost = df['transshipment_cost'].sum()
        network_total_cost = h1_total_cost + h2_total_cost + total_trans_cost

        h1_total_demand = df['demand_h1'].sum()
        h2_total_demand = df['demand_h2'].sum()
        h1_shortage_rate = (df['shortage_h1'].sum() / h1_total_demand * 100) if h1_total_demand > 0 else 0
        h2_shortage_rate = (df['shortage_h2'].sum() / h2_total_demand * 100) if h2_total_demand > 0 else 0
        h1_wastage_rate = (df['wastage_h1'].sum() / h1_total_demand * 100) if h1_total_demand > 0 else 0
        h2_wastage_rate = (df['wastage_h2'].sum() / h2_total_demand * 100) if h2_total_demand > 0 else 0

        total_trans_units = df['transferred_h1_to_h2'].sum() + df['transferred_h2_to_h1'].sum()

        seed_result = {
            'seed': seed,
            'h1_total_cost': h1_total_cost, 'h1_shortage_cost': h1_shortage_cost.sum(),
            'h1_shortage_rate': h1_shortage_rate, 'h1_wastage_cost': h1_wastage_cost.sum(),
            'h1_wastage_rate': h1_wastage_rate, 'h1_holding_cost': h1_holding_cost.sum(),
            'h1_order_cost': h1_order_cost.sum(),
            'h2_total_cost': h2_total_cost, 'h2_shortage_cost': h2_shortage_cost.sum(),
            'h2_shortage_rate': h2_shortage_rate, 'h2_wastage_cost': h2_wastage_cost.sum(),
            'h2_wastage_rate': h2_wastage_rate, 'h2_holding_cost': h2_holding_cost.sum(),
            'h2_order_cost': h2_order_cost.sum(),
            'network_total_cost': network_total_cost, 'total_trans_cost': total_trans_cost,
            'total_trans_units': total_trans_units,
            'trans_h1_to_h2': df['transferred_h1_to_h2'].sum(),
            'trans_h2_to_h1': df['transferred_h2_to_h1'].sum(),
        }
        all_seed_results.append(seed_result)

    return pd.DataFrame(all_seed_results)

# =============================================================================
# SENSITIVITY ANALYSIS DRIVER
# =============================================================================
def run_sensitivity_analysis():
    os.makedirs("sensitivity_results_no_trans", exist_ok=True)

    # --- Analysis 1: disruption_mult sensitivity ---
    disruption_mult_values = [1.0, 1.7, 2.0]
    disruption_results = []

    print("\n" + "="*80)
    print("SENSITIVITY ANALYSIS 1: DISRUPTION MULTIPLIER (Single-Agent PPO, No Transshipment)")
    print("="*80)

    for dm in disruption_mult_values:
        print(f"\n>>> Training with disruption_mult = {dm}")
        env_cfg = BASE_ENV_CONFIG.copy()
        env_cfg["disruption_mult"] = dm

        model_path = f"sensitivity_results_no_trans/no_trans_ppo_dm_{dm:.1f}.pt"

        best_reward = train(env_cfg, model_path, total_episodes=TOTAL_EPISODES)
        print(f"    Best 100-ep avg reward: {best_reward:.4f}")

        print(f"    Evaluating on 50 seeds...")
        results_df = evaluate_multi_seed(env_cfg, model_path, seeds=range(1, 51))

        if results_df is not None:
            results_df.to_csv(f"sensitivity_results_no_trans/results_dm_{dm:.1f}.csv", index=False)

            agg = {
                'disruption_mult': dm,
                'network_total_cost_mean': results_df['network_total_cost'].mean(),
                'network_total_cost_std': results_df['network_total_cost'].std(),
                'h1_shortage_rate_mean': results_df['h1_shortage_rate'].mean(),
                'h1_shortage_rate_std': results_df['h1_shortage_rate'].std(),
                'h2_shortage_rate_mean': results_df['h2_shortage_rate'].mean(),
                'h2_shortage_rate_std': results_df['h2_shortage_rate'].std(),
                'h1_wastage_rate_mean': results_df['h1_wastage_rate'].mean(),
                'h1_wastage_rate_std': results_df['h1_wastage_rate'].std(),
                'h2_wastage_rate_mean': results_df['h2_wastage_rate'].mean(),
                'h2_wastage_rate_std': results_df['h2_wastage_rate'].std(),
                'total_trans_cost_mean': results_df['total_trans_cost'].mean(),
                'total_trans_cost_std': results_df['total_trans_cost'].std(),
                'total_trans_units_mean': results_df['total_trans_units'].mean(),
                'total_trans_units_std': results_df['total_trans_units'].std(),
            }
            disruption_results.append(agg)

    # --- Analysis 2: wastage_cost / shortage_cost sensitivity ---
    cost_pairs = [(15, 15), (15, 45), (15, 60)]
    cost_results = []

    print("\n" + "="*80)
    print("SENSITIVITY ANALYSIS 2: WASTAGE_COST / SHORTAGE_COST RATIO (Single-Agent PPO, No Transshipment)")
    print("="*80)

    for wc, sc in cost_pairs:
        ratio = wc / sc
        print(f"\n>>> Training with wastage_cost={wc}, shortage_cost={sc} (ratio={ratio:.2f})")
        env_cfg = BASE_ENV_CONFIG.copy()
        env_cfg["wastage_cost"] = float(wc)
        env_cfg["shortage_cost"] = float(sc)

        model_path = f"sensitivity_results_no_trans/no_trans_ppo_wc{wc}_sc{sc}.pt"

        best_reward = train(env_cfg, model_path, total_episodes=TOTAL_EPISODES)
        print(f"    Best 100-ep avg reward: {best_reward:.4f}")

        print(f"    Evaluating on 50 seeds...")
        results_df = evaluate_multi_seed(env_cfg, model_path, seeds=range(1, 51))

        if results_df is not None:
            results_df.to_csv(f"sensitivity_results_no_trans/results_wc{wc}_sc{sc}.csv", index=False)

            agg = {
                'wastage_cost': wc,
                'shortage_cost': sc,
                'ratio': ratio,
                'network_total_cost_mean': results_df['network_total_cost'].mean(),
                'network_total_cost_std': results_df['network_total_cost'].std(),
                'h1_shortage_rate_mean': results_df['h1_shortage_rate'].mean(),
                'h1_shortage_rate_std': results_df['h1_shortage_rate'].std(),
                'h2_shortage_rate_mean': results_df['h2_shortage_rate'].mean(),
                'h2_shortage_rate_std': results_df['h2_shortage_rate'].std(),
                'h1_wastage_rate_mean': results_df['h1_wastage_rate'].mean(),
                'h1_wastage_rate_std': results_df['h1_wastage_rate'].std(),
                'h2_wastage_rate_mean': results_df['h2_wastage_rate'].mean(),
                'h2_wastage_rate_std': results_df['h2_wastage_rate'].std(),
                'total_trans_cost_mean': results_df['total_trans_cost'].mean(),
                'total_trans_cost_std': results_df['total_trans_cost'].std(),
                'total_trans_units_mean': results_df['total_trans_units'].mean(),
                'total_trans_units_std': results_df['total_trans_units'].std(),
            }
            cost_results.append(agg)

    # =============================================================================
    # PRINT SUMMARY TABLES
    # =============================================================================
    print("\n\n" + "="*100)
    print("SUMMARY TABLE 1: SENSITIVITY TO DISRUPTION MULTIPLIER (No Transshipment)")
    print("="*100)

    header1 = (f"{'disruption_mult':>15} | {'Net_Cost':>12} | {'H1_Short%':>12} | {'H2_Short%':>12} | "
               f"{'H1_Waste%':>12} | {'H2_Waste%':>12} | {'Trans_Cost':>12} | {'Trans_Units':>12}")
    print(header1)
    print("-" * 100)

    for r in disruption_results:
        line = (f"{r['disruption_mult']:>15.1f} | "
                f"{r['network_total_cost_mean']:>10.2f}±{r['network_total_cost_std']:<4.2f} | "
                f"{r['h1_shortage_rate_mean']:>10.2f}±{r['h1_shortage_rate_std']:<4.2f} | "
                f"{r['h2_shortage_rate_mean']:>10.2f}±{r['h2_shortage_rate_std']:<4.2f} | "
                f"{r['h1_wastage_rate_mean']:>10.2f}±{r['h1_wastage_rate_std']:<4.2f} | "
                f"{r['h2_wastage_rate_mean']:>10.2f}±{r['h2_wastage_rate_std']:<4.2f} | "
                f"{r['total_trans_cost_mean']:>10.2f}±{r['total_trans_cost_std']:<4.2f} | "
                f"{r['total_trans_units_mean']:>10.2f}±{r['total_trans_units_std']:<4.2f}")
        print(line)

    print("\n" + "="*100)
    print("SUMMARY TABLE 2: SENSITIVITY TO WASTAGE_COST / SHORTAGE_COST RATIO (No Transshipment)")
    print("="*100)

    header2 = (f"{'WC/SC':>10} | {'Net_Cost':>12} | {'H1_Short%':>12} | {'H2_Short%':>12} | "
               f"{'H1_Waste%':>12} | {'H2_Waste%':>12} | {'Trans_Cost':>12} | {'Trans_Units':>12}")
    print(header2)
    print("-" * 100)

    for r in cost_results:
        label = f"{r['wastage_cost']:.0f}/{r['shortage_cost']:.0f}"
        line = (f"{label:>10} | "
                f"{r['network_total_cost_mean']:>10.2f}±{r['network_total_cost_std']:<4.2f} | "
                f"{r['h1_shortage_rate_mean']:>10.2f}±{r['h1_shortage_rate_std']:<4.2f} | "
                f"{r['h2_shortage_rate_mean']:>10.2f}±{r['h2_shortage_rate_std']:<4.2f} | "
                f"{r['h1_wastage_rate_mean']:>10.2f}±{r['h1_wastage_rate_std']:<4.2f} | "
                f"{r['h2_wastage_rate_mean']:>10.2f}±{r['h2_wastage_rate_std']:<4.2f} | "
                f"{r['total_trans_cost_mean']:>10.2f}±{r['total_trans_cost_std']:<4.2f} | "
                f"{r['total_trans_units_mean']:>10.2f}±{r['total_trans_units_std']:<4.2f}")
        print(line)

    pd.DataFrame(disruption_results).to_csv("sensitivity_results_no_trans/summary_disruption_mult.csv", index=False)
    pd.DataFrame(cost_results).to_csv("sensitivity_results_no_trans/summary_cost_ratio.csv", index=False)
    print("\nSummary tables saved to sensitivity_results_no_trans/")


if __name__ == "__main__":
    run_sensitivity_analysis()


SENSITIVITY ANALYSIS 1: DISRUPTION MULTIPLIER (Single-Agent PPO, No Transshipment)

>>> Training with disruption_mult = 1.0


Training: 100%|██████████| 2000/2000 [02:45<00:00, 12.12it/s]


    Best 100-ep avg reward: -60.8966
    Evaluating on 50 seeds...


Evaluating: 100%|██████████| 50/50 [00:02<00:00, 18.62it/s]



>>> Training with disruption_mult = 1.7


...


>>> Training with wastage_cost=15, shortage_cost=60 (ratio=0.25)


Training: 100%|██████████| 2000/2000 [02:51<00:00, 11.63it/s]


    Best 100-ep avg reward: -85.5284
    Evaluating on 50 seeds...


Evaluating: 100%|██████████| 50/50 [00:02<00:00, 20.36it/s]



SUMMARY TABLE 1: SENSITIVITY TO DISRUPTION MULTIPLIER (No Transshipment)
disruption_mult |     Net_Cost |    H1_Short% |    H2_Short% |    H1_Waste% |    H2_Waste% |   Trans_Cost |  Trans_Units
----------------------------------------------------------------------------------------------------
            1.0 |    5874.29±789.28 |       2.82±1.59 |       1.83±1.54 |       8.74±4.53 |       7.24±4.71 |       0.00±0.00 |       0.00±0.00
            1.7 |    6829.55±1140.78 |       5.67±3.27 |       3.62±2.94 |       6.55±3.52 |       8.12±4.76 |       0.00±0.00 |       0.00±0.00
            2.0 |    7833.19±1602.53 |       6.30±4.31 |       4.43±4.47 |       8.70±4.40 |       8.49±5.07 |       0.00±0.00 |       0.00±0.00

SUMMARY TABLE 2: SENSITIVITY TO WASTAGE_COST / SHORTAGE_COST RATIO (No Transshipment)
     WC/SC |     Net_Cost |    H1_Short% |    H2_Short% |    H1_Waste% |    H2_Waste% |   Trans_Cost |  Trans_Units
------------------------------------------------------------------